In [0]:
import pandas as pd

# Read Silver Delta table
df_silver = spark.table("silver_hospital_general").toPandas()

# GOLD TABLE 1 - Average rating by state
gold_by_state = df_silver[df_silver['overall_rating'].notna()].groupby('state').agg(
    avg_rating=('overall_rating', 'mean'),
    hospital_count=('facility_id', 'count'),
    rated_count=('overall_rating', 'count')
).reset_index().sort_values('avg_rating', ascending=False).round(2)

# GOLD TABLE 2 - Average rating by hospital type
gold_by_type = df_silver[df_silver['overall_rating'].notna()].groupby('hospital_type').agg(
    avg_rating=('overall_rating', 'mean'),
    hospital_count=('facility_id', 'count')
).reset_index().sort_values('avg_rating', ascending=False).round(2)

# GOLD TABLE 3 - Average rating by ownership
gold_by_ownership = df_silver[df_silver['overall_rating'].notna()].groupby('ownership').agg(
    avg_rating=('overall_rating', 'mean'),
    hospital_count=('facility_id', 'count')
).reset_index().sort_values('avg_rating', ascending=False).round(2)

# GOLD TABLE 4 - Top 20 rated hospitals
gold_top_hospitals = df_silver[df_silver['overall_rating'] == 5][
    ['facility_id', 'facility_name', 'city', 'state', 'hospital_type', 'ownership', 'overall_rating']
].head(20)

print(f"Gold by state: {len(gold_by_state)} rows")
print(f"Gold by type: {len(gold_by_type)} rows")
print(f"Gold by ownership: {len(gold_by_ownership)} rows")
print(f"Top rated hospitals: {len(gold_top_hospitals)} rows")

print("\nTop 5 States by Average Rating:")
display(gold_by_state.head())

print("\nRating by Hospital Type:")
display(gold_by_type)

Gold by state: 5 rows
Gold by type: 3 rows
Gold by ownership: 11 rows
Top rated hospitals: 20 rows

Top 5 States by Average Rating:


state,avg_rating,hospital_count,rated_count
CA,3.15,167,167
AZ,3.11,55,55
AK,2.9,10,10
AR,2.84,44,44
AL,2.71,59,59



Rating by Hospital Type:


hospital_type,avg_rating,hospital_count
VA Hospitals,4.36,11
Critical Access Hospitals,3.42,12
Acute Care Hospitals,2.96,312


In [0]:
# Write all Gold tables to Delta
spark.createDataFrame(gold_by_state).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_hospital_by_state")

spark.createDataFrame(gold_by_type).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_hospital_by_type")

spark.createDataFrame(gold_by_ownership).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_hospital_by_ownership")

spark.createDataFrame(gold_top_hospitals).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_top_rated_hospitals")

print("All Gold Delta tables created successfully!")
print("  gold_hospital_by_state")
print("  gold_hospital_by_type")
print("  gold_hospital_by_ownership")
print("  gold_top_rated_hospitals")

All Gold Delta tables created successfully!
  gold_hospital_by_state
  gold_hospital_by_type
  gold_hospital_by_ownership
  gold_top_rated_hospitals


In [0]:
# Verify hospital_type standardization
display(df_silver['hospital_type'].value_counts())

hospital_type
Acute Care Hospitals                  359
Critical Access Hospitals              67
Psychiatric                            42
VA Hospitals                           12
Rural Emergency Hospital                8
Acute Care - Department of Defense      7
Childrens                               5
Name: count, dtype: int64